In [2]:
import pandas as pd

demand_sales_clean = pd.read_excel(r"C:\Users\Vasu Bhardwaj\OneDrive\Desktop\Healthcare-Supply-Chain-Inventory-Analytics\data\raw\demand_sales.xlsx"
)

print("Demand & Sales loaded successfully.")
print("Rows:", demand_sales_clean.shape[0])
print("Columns:", demand_sales_clean.shape[1])

Demand & Sales loaded successfully.
Rows: 14218
Columns: 16


In [2]:
# Basic text cleaning
text_cols = [
    "Order_ID",
    "Patient_ID",
    "Dept",
    "Specialisation",
    "Medicine_ID",
    "DrugName",
    "Formulation",
    "Order_Status"
]

for col in text_cols:
    demand_sales_clean[col] = (
        demand_sales_clean[col]
        .astype(str)
        .str.strip()
    )

# Standardize blank text values
demand_sales_clean[text_cols] = demand_sales_clean[text_cols].replace(
    "", pd.NA
)

# Handle known missing formulation values
demand_sales_clean["Formulation"] = (
    demand_sales_clean["Formulation"]
    .fillna("Unknown")
)

# Ensure date columns are proper datetime
demand_sales_clean["Order_Date"] = pd.to_datetime(
    demand_sales_clean["Order_Date"],
    errors="coerce"
)

demand_sales_clean["Delivery_Date"] = pd.to_datetime(
    demand_sales_clean["Delivery_Date"],
    errors="coerce"
)

print("Initial cleaning completed.")
print("Rows:", demand_sales_clean.shape[0])
print("Columns:", demand_sales_clean.shape[1])
print("\nRemaining missing values:")
print(demand_sales_clean.isna().sum())

Initial cleaning completed.
Rows: 14218
Columns: 16

Remaining missing values:
Order_ID             0
Order_Date           0
Patient_ID           0
Dept                 0
Specialisation       0
Medicine_ID          0
DrugName          1668
Formulation          0
Quantity             0
ReturnQuantity       0
Final_Cost           0
Final_Sales          0
RtnMRP               0
Order_Status         0
Delivery_Date      633
Lead_Time_Days       0
dtype: int64


In [3]:
# ==========================================
# BUSINESS RULE CLEANING
# ==========================================

# 1. Handle missing DrugName
demand_sales_clean["DrugName"] = (
    demand_sales_clean["DrugName"]
    .fillna("Unknown")
)

# 2. Recalculate Lead Time from Order Date and Delivery Date
#    Only where Delivery_Date is available
valid_delivery = demand_sales_clean["Delivery_Date"].notna()

demand_sales_clean.loc[valid_delivery, "Lead_Time_Days"] = (
    demand_sales_clean.loc[valid_delivery, "Delivery_Date"]
    - demand_sales_clean.loc[valid_delivery, "Order_Date"]
).dt.days

# 3. Keep Lead Time as integer
demand_sales_clean["Lead_Time_Days"] = (
    demand_sales_clean["Lead_Time_Days"]
    .astype(int)
)

# 4. Validate key business rules
print("=== CLEANING VALIDATION ===")

print(
    "Missing DrugName:",
    demand_sales_clean["DrugName"].isna().sum()
)

print(
    "Missing Delivery Date - Cancelled:",
    demand_sales_clean.loc[
        demand_sales_clean["Order_Status"] == "Cancelled",
        "Delivery_Date"
    ].isna().sum()
)

print(
    "Negative Lead Time:",
    (demand_sales_clean["Lead_Time_Days"] < 0).sum()
)

print(
    "Returned orders with ReturnQuantity = 0:",
    (
        (demand_sales_clean["Order_Status"] == "Returned") &
        (demand_sales_clean["ReturnQuantity"] == 0)
    ).sum()
)

print(
    "Returned orders with Final_Sales != 0:",
    (
        (demand_sales_clean["Order_Status"] == "Returned") &
        (demand_sales_clean["Final_Sales"] != 0)
    ).sum()
)

=== CLEANING VALIDATION ===
Missing DrugName: 0
Missing Delivery Date - Cancelled: 633
Negative Lead Time: 0
Returned orders with ReturnQuantity = 0: 0
Returned orders with Final_Sales != 0: 0


In [4]:
# ==========================================
# FINAL VALIDATION & EXPORT
# ==========================================

print("=== FINAL DATA VALIDATION ===")

print("Rows:", demand_sales_clean.shape[0])
print("Columns:", demand_sales_clean.shape[1])
print("Duplicate rows:", demand_sales_clean.duplicated().sum())
print("Total missing values:", demand_sales_clean.isna().sum().sum())

# Key business validations
print(
    "Invalid Lead Time:",
    (demand_sales_clean["Lead_Time_Days"] < 0).sum()
)

print(
    "Returned orders with sales > 0:",
    (
        (demand_sales_clean["Order_Status"] == "Returned") &
        (demand_sales_clean["Final_Sales"] > 0)
    ).sum()
)

print(
    "Cancelled orders with Delivery Date:",
    (
        (demand_sales_clean["Order_Status"] == "Cancelled") &
        demand_sales_clean["Delivery_Date"].notna()
    ).sum()
)

# Create cleaned folder if it doesn't exist
import os

cleaned_path = (
    r"C:\Users\Vasu Bhardwaj\OneDrive\Desktop"
    r"\Healthcare-Supply-Chain-Inventory-Analytics\data\cleaned"
)

os.makedirs(cleaned_path, exist_ok=True)

# Export cleaned dataset as CSV
output_file = os.path.join(
    cleaned_path,
    "demand_sales_cleaned.csv"
)

demand_sales_clean.to_csv(
    output_file,
    index=False
)

print("\nCleaned Demand & Sales dataset exported successfully.")
print("File:", output_file)

=== FINAL DATA VALIDATION ===
Rows: 14218
Columns: 16
Duplicate rows: 0
Total missing values: 633
Invalid Lead Time: 0
Returned orders with sales > 0: 0
Cancelled orders with Delivery Date: 0

Cleaned Demand & Sales dataset exported successfully.
File: C:\Users\Vasu Bhardwaj\OneDrive\Desktop\Healthcare-Supply-Chain-Inventory-Analytics\data\cleaned\demand_sales_cleaned.csv


In [5]:
# ==========================================
# FINAL BUSINESS VALIDATION
# ==========================================

expected_missing = (
    (demand_sales_clean["Order_Status"] == "Cancelled") &
    demand_sales_clean["Delivery_Date"].isna()
).sum()

unexpected_missing = (
    demand_sales_clean.isna().sum().sum()
    - expected_missing
)

print("=== FINAL VALIDATION ===")

print("Rows:", demand_sales_clean.shape[0])
print("Columns:", demand_sales_clean.shape[1])
print("Duplicate rows:", demand_sales_clean.duplicated().sum())

print("Expected missing Delivery_Date:", expected_missing)
print("Unexpected missing values:", unexpected_missing)

print(
    "Invalid Lead Time:",
    (demand_sales_clean["Lead_Time_Days"] < 0).sum()
)

print(
    "Returned orders with sales > 0:",
    (
        (demand_sales_clean["Order_Status"] == "Returned") &
        (demand_sales_clean["Final_Sales"] > 0)
    ).sum()
)

print(
    "Cancelled orders with Delivery Date:",
    (
        (demand_sales_clean["Order_Status"] == "Cancelled") &
        demand_sales_clean["Delivery_Date"].notna()
    ).sum()
)

print(
    "Unknown DrugName:",
    (demand_sales_clean["DrugName"] == "Unknown").sum()
)

=== FINAL VALIDATION ===
Rows: 14218
Columns: 16
Duplicate rows: 0
Expected missing Delivery_Date: 633
Unexpected missing values: 0
Invalid Lead Time: 0
Returned orders with sales > 0: 0
Cancelled orders with Delivery Date: 0
Unknown DrugName: 1668


In [6]:
# ==========================================
# CONSUMPTION FACILITY - LOAD
# ==========================================

consumption_facility_clean = pd.read_excel(
    r"C:\Users\Vasu Bhardwaj\OneDrive\Desktop"
    r"\Healthcare-Supply-Chain-Inventory-Analytics\data\consumption_facility.xlsx"
)

print("Consumption Facility loaded successfully.")
print("Rows:", consumption_facility_clean.shape[0])
print("Columns:", consumption_facility_clean.shape[1])

Consumption Facility loaded successfully.
Rows: 11111
Columns: 9


In [7]:
# ==========================================
# CONSUMPTION FACILITY - CLEAN & EXPORT
# ==========================================

# Text columns
text_cols_cf = [
    "Patient_ID",
    "Region",
    "Medicine_ID",
    "DrugName",
    "Supplies_Used"
]

# Standardize text fields
for col in text_cols_cf:
    consumption_facility_clean[col] = (
        consumption_facility_clean[col]
        .astype(str)
        .str.strip()
    )

# Numeric columns
numeric_cols_cf = [
    "Daily_Consumption_Units",
    "Out_of_Stock_Days",
    "Wastage_Units",
    "Bed_Days"
]

# Validate numeric values
negative_counts_cf = (
    consumption_facility_clean[numeric_cols_cf] < 0
).sum()

# Remove exact duplicates if any
duplicates_cf = consumption_facility_clean.duplicated().sum()

if duplicates_cf > 0:
    consumption_facility_clean = (
        consumption_facility_clean.drop_duplicates()
    )

# Create cleaned folder
import os

cleaned_path = (
    r"C:\Users\Vasu Bhardwaj\OneDrive\Desktop"
    r"\Healthcare-Supply-Chain-Inventory-Analytics\data\cleaned"
)

os.makedirs(cleaned_path, exist_ok=True)

# Export CSV
output_file_cf = os.path.join(
    cleaned_path,
    "consumption_facility_cleaned.csv"
)

consumption_facility_clean.to_csv(
    output_file_cf,
    index=False
)

# Final validation
print("=== CONSUMPTION FACILITY VALIDATION ===")
print("Rows:", consumption_facility_clean.shape[0])
print("Columns:", consumption_facility_clean.shape[1])
print("Duplicate rows:", consumption_facility_clean.duplicated().sum())
print(
    "Missing values:",
    consumption_facility_clean.isna().sum().sum()
)
print("\nNegative values:")
print(negative_counts_cf)

print("\nCleaned Consumption Facility exported successfully.")
print("File:", output_file_cf)

=== CONSUMPTION FACILITY VALIDATION ===
Rows: 11111
Columns: 9
Duplicate rows: 0
Missing values: 0

Negative values:
Daily_Consumption_Units    0
Out_of_Stock_Days          0
Wastage_Units              0
Bed_Days                   0
dtype: int64

Cleaned Consumption Facility exported successfully.
File: C:\Users\Vasu Bhardwaj\OneDrive\Desktop\Healthcare-Supply-Chain-Inventory-Analytics\data\cleaned\consumption_facility_cleaned.csv


In [8]:
# ==========================================
# INVENTORY STOCK - LOAD
# ==========================================

inventory_stock_clean = pd.read_excel(
    r"C:\Users\Vasu Bhardwaj\OneDrive\Desktop"
    r"\Healthcare-Supply-Chain-Inventory-Analytics\data\inventory_stock.xlsx"
)

print("Inventory Stock loaded successfully.")
print("Rows:", inventory_stock_clean.shape[0])
print("Columns:", inventory_stock_clean.shape[1])

Inventory Stock loaded successfully.
Rows: 841
Columns: 16


In [9]:
# ==========================================
# INVENTORY STOCK - CLEAN & EXPORT
# ==========================================

# Text columns
text_cols_inv = [
    "Medicine_ID",
    "DrugName",
    "Formulation",
    "Category",
    "Batch_No",
    "Vendor_ID"
]

# Standardize text fields
for col in text_cols_inv:
    inventory_stock_clean[col] = (
        inventory_stock_clean[col]
        .astype(str)
        .str.strip()
    )

# Date columns
date_cols_inv = [
    "Manufacture_Date",
    "Expiry_Date"
]

for col in date_cols_inv:
    inventory_stock_clean[col] = pd.to_datetime(
        inventory_stock_clean[col],
        errors="coerce"
    )

# Numeric columns
numeric_cols_inv = [
    "Current_Stock",
    "Min_Required",
    "Max_Capacity",
    "Unit_Cost",
    "Avg_Usage_Per_Day",
    "Restock_Lead_Time_Days",
    "Reorder_Level",
    "Stock_Cover_Days"
]

# Validate negative values
negative_counts_inv = (
    inventory_stock_clean[numeric_cols_inv] < 0
).sum()

# Remove exact duplicates if any
duplicates_inv = inventory_stock_clean.duplicated().sum()

if duplicates_inv > 0:
    inventory_stock_clean = inventory_stock_clean.drop_duplicates()

# Create cleaned folder
import os

cleaned_path = (
    r"C:\Users\Vasu Bhardwaj\OneDrive\Desktop"
    r"\Healthcare-Supply-Chain-Inventory-Analytics\data\cleaned"
)

os.makedirs(cleaned_path, exist_ok=True)

# Export CSV
output_file_inv = os.path.join(
    cleaned_path,
    "inventory_stock_cleaned.csv"
)

inventory_stock_clean.to_csv(
    output_file_inv,
    index=False
)

# Final validation
print("=== INVENTORY STOCK VALIDATION ===")
print("Rows:", inventory_stock_clean.shape[0])
print("Columns:", inventory_stock_clean.shape[1])
print("Duplicate rows:", inventory_stock_clean.duplicated().sum())
print(
    "Missing values:",
    inventory_stock_clean.isna().sum().sum()
)

print("\nNegative values:")
print(negative_counts_inv)

print(
    "\nCurrent Stock > Max Capacity:",
    (
        inventory_stock_clean["Current_Stock"]
        > inventory_stock_clean["Max_Capacity"]
    ).sum()
)

print(
    "Expiry before Manufacture:",
    (
        inventory_stock_clean["Expiry_Date"]
        < inventory_stock_clean["Manufacture_Date"]
    ).sum()
)

print("\nCleaned Inventory Stock exported successfully.")
print("File:", output_file_inv)

=== INVENTORY STOCK VALIDATION ===
Rows: 841
Columns: 16
Duplicate rows: 0
Missing values: 0

Negative values:
Current_Stock             0
Min_Required              0
Max_Capacity              0
Unit_Cost                 0
Avg_Usage_Per_Day         0
Restock_Lead_Time_Days    0
Reorder_Level             0
Stock_Cover_Days          0
dtype: int64

Current Stock > Max Capacity: 0
Expiry before Manufacture: 0

Cleaned Inventory Stock exported successfully.
File: C:\Users\Vasu Bhardwaj\OneDrive\Desktop\Healthcare-Supply-Chain-Inventory-Analytics\data\cleaned\inventory_stock_cleaned.csv


In [10]:
# ==========================================
# SUPPLIER PROCUREMENT - LOAD
# ==========================================

supplier_procurement_clean = pd.read_excel(
    r"C:\Users\Vasu Bhardwaj\OneDrive\Desktop"
    r"\Healthcare-Supply-Chain-Inventory-Analytics\data\supplier_procurement.xlsx"
)

print("Supplier Procurement loaded successfully.")
print("Rows:", supplier_procurement_clean.shape[0])
print("Columns:", supplier_procurement_clean.shape[1])

Supplier Procurement loaded successfully.
Rows: 538
Columns: 9


In [11]:
# ==========================================
# SUPPLIER PROCUREMENT - CLEAN & EXPORT
# ==========================================

# Text columns
text_cols_sup = [
    "Supplier_ID",
    "Supplier_Name",
    "Region"
]

# Standardize text fields
for col in text_cols_sup:
    supplier_procurement_clean[col] = (
        supplier_procurement_clean[col]
        .astype(str)
        .str.strip()
    )

# Date columns
date_cols_sup = [
    "Last_Order_Date",
    "Next_Delivery_Date"
]

for col in date_cols_sup:
    supplier_procurement_clean[col] = pd.to_datetime(
        supplier_procurement_clean[col],
        errors="coerce"
    )

# Remove exact duplicates if any
duplicates_sup = supplier_procurement_clean.duplicated().sum()

if duplicates_sup > 0:
    supplier_procurement_clean = (
        supplier_procurement_clean.drop_duplicates()
    )

# Create cleaned folder
import os

cleaned_path = (
    r"C:\Users\Vasu Bhardwaj\OneDrive\Desktop"
    r"\Healthcare-Supply-Chain-Inventory-Analytics\data\cleaned"
)

os.makedirs(cleaned_path, exist_ok=True)

# Export CSV
output_file_sup = os.path.join(
    cleaned_path,
    "supplier_procurement_cleaned.csv"
)

supplier_procurement_clean.to_csv(
    output_file_sup,
    index=False
)

# ==========================================
# FINAL VALIDATION
# ==========================================

print("=== SUPPLIER PROCUREMENT VALIDATION ===")

print("Rows:", supplier_procurement_clean.shape[0])
print("Columns:", supplier_procurement_clean.shape[1])

print(
    "Duplicate rows:",
    supplier_procurement_clean.duplicated().sum()
)

print(
    "Missing values:",
    supplier_procurement_clean.isna().sum().sum()
)

print(
    "Negative Avg Lead Time:",
    (supplier_procurement_clean["Avg_Lead_Time_Days"] < 0).sum()
)

print(
    "Negative Cost:",
    (supplier_procurement_clean["Cost_Per_Item"] < 0).sum()
)

print(
    "Invalid Delivery Dates:",
    (
        supplier_procurement_clean["Next_Delivery_Date"]
        < supplier_procurement_clean["Last_Order_Date"]
    ).sum()
)

print(
    "Reliability Score outside 1-5:",
    (
        (supplier_procurement_clean["Reliability_Score"] < 1) |
        (supplier_procurement_clean["Reliability_Score"] > 5)
    ).sum()
)

print(
    "On-Time Delivery Rate outside 0-100:",
    (
        (supplier_procurement_clean["On_Time_Delivery_Rate"] < 0) |
        (supplier_procurement_clean["On_Time_Delivery_Rate"] > 100)
    ).sum()
)

print("\nCleaned Supplier Procurement exported successfully.")
print("File:", output_file_sup)

=== SUPPLIER PROCUREMENT VALIDATION ===
Rows: 538
Columns: 9
Duplicate rows: 0
Missing values: 0
Negative Avg Lead Time: 0
Negative Cost: 0
Invalid Delivery Dates: 0
Reliability Score outside 1-5: 0
On-Time Delivery Rate outside 0-100: 0

Cleaned Supplier Procurement exported successfully.
File: C:\Users\Vasu Bhardwaj\OneDrive\Desktop\Healthcare-Supply-Chain-Inventory-Analytics\data\cleaned\supplier_procurement_cleaned.csv


In [3]:
# Check unusually long DrugName values
drugname_length = demand_sales_clean["DrugName"].astype("string").str.len()

print("Maximum DrugName length:", drugname_length.max())

demand_sales_clean.loc[
    drugname_length > 100,
    ["Medicine_ID", "DrugName"]
].head(20)

Maximum DrugName length: 210


,Medicine_ID,DrugName
41,M28854D7C,SODIUM CHLORIDE 600MG + SODIUM LACTATE 320MG ...
87,MB9018922,BACILLUS MESENTERICUS 1 MILLION CELLS + CLOST...
144,M6E491D65,SODIUM CHLORIDE 600MG + SODIUM LACTATE 320MG ...
180,MC8E620FF,SODIUM CHLORIDE 2.6GM + POTASSIUM CHLORIDE 1....
210,MF00C5866,SODIUM CHLORIDE 600MG + SODIUM LACTATE 320MG ...
265,MCF99B44A,CALCIUM PANTOTHENATE 50MG + CYANOCOBALAMIN 15...
326,M8DFE291A,SODIUM CHLORIDE 600MG + SODIUM LACTATE 320MG ...
411,M2267343C,SODIUM CHLORIDE 600MG + SODIUM LACTATE 320MG ...
439,M224F2EAE,SODIUM CHLORIDE 600MG + SODIUM LACTATE 320MG ...
466,M5ABD6E53,SODIUM CHLORIDE 600MG + SODIUM LACTATE 320MG ...


In [4]:
# Normalize DrugName whitespace and remove embedded line breaks
demand_sales_clean["DrugName"] = (
    demand_sales_clean["DrugName"]
    .astype("string")
    .str.replace(r"\s+", " ", regex=True)
    .str.strip()
)

# Validate DrugName lengths after cleaning
drugname_length = demand_sales_clean["DrugName"].str.len()

print("Maximum DrugName length:", drugname_length.max())
print("DrugName > 200 characters:", (drugname_length > 200).sum())

demand_sales_clean.loc[
    drugname_length > 200,
    ["Medicine_ID", "DrugName"]
].head(10)

Maximum DrugName length: 209
DrugName > 200 characters: 1


,Medicine_ID,DrugName
6155,M769313A9,VITAMIN B12 + FOLIC ACID + IRON HYDROCHLORIDE\...
